# 応用編10

V3.5新機能:SPV(Simplified Payment Verification)機能

このノートブックでは、APIバージョン3.5から利用できるSPV機能について、使い方を説明します。 


## SPV(Simplified Payment Verification)機能の概要

SPVは、ブロックチェーン全体のデータをダウンロードすることなく、一部のデータのみを使った暗号学的な計算によって、特定のトランザクションがブロックチェーンに正しく記録されていることを証明する検証方法です。検証範囲を特定のトランザクションに絞ることで計算時間は比較的短いという特徴があります。  
BC+では、このSPVをトランザクションの発行時にリアルタイムに行う機能をクライアント・ライブラリに実装しています。

## 準備

設定やライブラリを読み込みます。  
特に、事前に設定したピア構成情報を表すJSON文字列を変数cnfstrに読み込みます。

In [1]:
var { api } = await import('../lib/load-blockchain-api.mjs');
var { domain, cnfstr } = await import('../lib/load-config.mjs');
var { adminWallet, rpc } = await import('../lib/notebook-util.mjs');

## ピア構成情報の最新化

ピア構成情報(cnfstr)から、ピア構成情報オブジェクトpeerscnfを得ます。

In [2]:
var peerscnf = await api.loadPeersCnf(cnfstr);

ブロックチェーンに問い合わせ、peerscnfを最新に更新します。

In [3]:
var peerscnf = await rpc.updatePeersCnf(peerscnf);

## リアルタイムSPVを有効にする

ピア構成情報オブジェクトをrpcにセットして、リアルタイムSPVを有効にします。

In [4]:
rpc.setSPV(peerscnf);

## トランザクションの発行とSPVの自動実行

rpcを利用してトランザクションを発行すると、自動的にSPVの検証が実行されます。  
返り値respのspvプロパティの値が2となる場合、SPVの検証が成功しています。  
なお、SPVの検証により何らかの改ざんが検出された場合には、rpc.call()はエラーをスローします。

In [5]:
var resp = await rpc.call(adminWallet, '_eval@'+domain, { text: '3+4' });
console.log(resp);

{
  txno: 703030,
  txid: 'xTwD3CZh5ax8YSJguZaiynwdYSaCkhZyH4i5VVvMRm8Mt',
  status: 'ok',
  value: 7,
  blockref: { no: 74134, hash: 'tRmji3pSmhxKjGxgV2AYLnzKaeiuVHaq5u3WQrUUawc=' },
  spv: 2
}


リアルタイムSPVを無効にするには、setSPV()を空打ちします。  
リアルタイムSPVが無効な場合、返り値respのspvプロパティは出力されません。

In [6]:
rpc.setSPV();
var resp = await rpc.call(adminWallet, '_eval@'+domain, { text: '3+4' });
console.log(resp);

{
  txno: 703031,
  txid: 'xUUTQDfZgEvYr9jD4r5SvCiLNSWXuNT9F3Kg6jxD6J4yPB',
  status: 'ok',
  value: 7
}
